## Pydantic

In [1]:
import os
from langchain.chat_models import init_chat_model
from dotenv import  load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:llama-3.3-70b-versatile")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002BD907ED2B0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002BD907EDD30>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [2]:
from pydantic import  BaseModel,Field

class Movie(BaseModel):
   title:str = Field(description="Title of the movie")
   year:int = Field(description="This year movie was released")
   director:str = Field(description="Director of the movie")
   rating:float = Field(description="Movie rating out of 10")

In [3]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002BD907ED2B0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002BD907EDD30>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'Title of the movie', 'type': 'string'}, 'year': {'description': 'This year movie was released', 'type': 'integer'}, 'director': {'description': 'Director of the movie', 'type': 'string'}, 'rating': {'description': 'Movie rating out of 10'

In [4]:
response = model_with_structure.invoke("provide detals about the movie Inception")

print(response)

title='Inception' year=2010 director='Christopher Nolan' rating=8.5


In [5]:
model.invoke("provide detals about the movie Inception")

AIMessage(content='**Inception (2010)**\n\nInception is a science fiction action film written, co-produced, and directed by Christopher Nolan. The movie explores the concept of shared dreaming, where a team of thieves navigates the subconscious mind to plant an idea instead of stealing one.\n\n**Plot**\n\nThe film follows Cobb (Leonardo DiCaprio), a skilled thief who specializes in entering people\'s dreams and stealing their secrets. Cobb is hired by a wealthy businessman named Saito (Ken Watanabe) to perform a task known as "inception" - planting an idea in someone\'s mind instead of stealing one.\n\nSaito wants Cobb to convince Robert Fischer (Cillian Murphy), the son of a dying business magnate, to dissolve his father\'s company. In return, Saito promises to clear Cobb\'s name, which is wanted by the authorities, and allow him to return to the United States to see his children.\n\nCobb assembles a team of experts to help him with the task:\n\n1. Arthur (Joseph Gordon-Levitt), a poi

### Message output alongside parsed structure

In [6]:
from pydantic import  BaseModel,Field

class Movie(BaseModel):
   """ A movie with details """
   title:str = Field(...,description="Title of the movie")
   year:int = Field(...,description="This year movie was released")
   director:str = Field(...,description="Director of the movie")
   rating:float = Field(...,description="Movie rating out of 10")

model_with_structered_output = model.with_structured_output(Movie,include_raw=True)
response = model_with_structered_output.invoke("Provide details about movie Inception")
print(response)

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '9df13ar7j', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.5,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 284, 'total_tokens': 316, 'completion_time': 0.05545449, 'completion_tokens_details': None, 'prompt_time': 0.015350929, 'prompt_tokens_details': None, 'queue_time': 0.050847681, 'total_time': 0.070805419}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_f8b414701e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e698c-5866-7852-bcb6-2cd73b33f91a-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Christopher Nolan', 'rating': 8.5, 'title': 'Inception', 'year': 2010}, 'id': '9df13ar7j', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 284, 'output_tokens': 32, 

### Nested Structer

In [7]:
class Actor(BaseModel):
    name : str
    role : str

class MovieDetails(BaseModel):
    title : str
    year : str
    cast : list[Actor]
    genres : list[str]
    budget : float = Field(description="Budget in million USD")

model_with_structered_output = model.with_structured_output(MovieDetails)
response = model_with_structered_output.invoke("Provide details about movie Inception")
print(response)

title='Inception' year='2010' cast=[Actor(name='Leonardo DiCaprio', role='Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur')] genres=['Action', 'Sci-Fi'] budget=160.0


## TypedDict

In [8]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
  title : Annotated[str,...,"the title of the movie"]  
  year : Annotated[int,...,"the year the movie was released"]  
  director : Annotated[str,...,"the director of the movie"]  
  rating : Annotated[float,...,"the movie rating out of 10"]  


model_withtypedict = model.with_structured_output(MovieDict)
response = model_withtypedict.invoke("please provide the detail of movie avenger")
response

{'director': 'Joss Whedon', 'rating': 8.1, 'title': 'Avenger', 'year': 2012}

In [12]:

class Actor(TypedDict):
    name: str
    role: str



class MovieDetails(TypedDict):
    title: str
    year: str
    cast: list[Actor]
    genres: list[str]
    budget: float | str | None



model_with_structured_output = model.with_structured_output(
    MovieDetails
)

response = model_with_structured_output.invoke(
    "Provide details about movie Avengers"
)

response

{'budget': '220 million',
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'title': 'Avengers',
 'year': '2012'}

## ClassData

In [18]:
import os
from pydantic import BaseModel
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

class ContactInfo(BaseModel):
   """ Contact Information for a person """
   name:str = Field(description="The name of the person")
   email:str = Field(...,description="The email of th person")
   phone:str = Field(...,description="The phone of the person")

model = init_chat_model(
    model="groq:llama-3.3-70b-versatile"
)

agent = create_agent(
    model=model,
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Extract contact information from Saif ,abc@gmail.com , 03352793857"
        }
    ]
})

print(result["structured_response"])
result

name='Saif' email='abc@gmail.com' phone='03352793857'


{'messages': [HumanMessage(content='Extract contact information from Saif ,abc@gmail.com , 03352793857', additional_kwargs={}, response_metadata={}, id='cfaea17e-ed3c-4fa7-814a-9b4c9902dcde'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'q92dvk437', 'function': {'arguments': '{"email":"abc@gmail.com","name":"Saif","phone":"03352793857"}', 'name': 'ContactInfo'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 282, 'total_tokens': 312, 'completion_time': 0.06310683, 'completion_tokens_details': None, 'prompt_time': 0.014795947, 'prompt_tokens_details': None, 'queue_time': 0.047663279, 'total_time': 0.077902777}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_ba38bbab80', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e69c9-74b7-7a81-925a-a37fb16f1404-0', tool_calls=[{'name': 'ContactInfo', 'args': {'email': 'abc@gmail.com',